# G — Entailment-based verification (GPU)

Replaces the similarity matcher in notebook F, which returned `not_in_curriculum` for 96%
of claims and could not be rescued by thresholding.

### Why similarity failed and entailment doesn't

Cosine similarity answers *"are this claim and this fact about the same thing?"* — which is
not verification. Every fact about `উচ্চ রক্তচাপ` scores alike against a claim about
`উচ্চ রক্তচাপ`, whether the claim is right or wrong.

Entailment answers the actual question. On constructed Bangla pairs, mDeBERTa scores
`মানবদেহে তিন ধরনের রক্তকোষ` against `রক্তকোষ পাঁচ প্রকার` as **contradiction 0.80** —
the case similarity scores as a near-match.

This is also the established paradigm: PGR (Findings of EMNLP 2025) describes prior KG
fact-verification work as *"employing natural language inference models to predict the
entailment between the evidence and the claim."*

### Design

1. **Retrieve** candidate facts per claim — entity links plus LaBSE top-k. Similarity is
   demoted to what it is good at: narrowing 2,248 facts to a handful.
2. **Verbalise** each triple into a Bangla sentence. NLI models are trained on sentence
   pairs; `রক্তকোষ প্রকার অণুচক্রিকা` is not a sentence.
3. **Entail** each (fact, claim) pair and aggregate: any entailment → supported; else any
   contradiction → contradicted; else not-in-curriculum.

Known limitation, measured rather than assumed: the model missed a verb antonym
(`অক্সিজেন ত্যাগ করে` vs `অক্সিজেন গ্রহণ করে` → neutral 0.66), so some contradictions will
read as neutral and land in not-in-curriculum.

**Settings:** GPU `NvidiaTeslaT4`, Internet on, dataset `bangla-biology-kg`. Reuses run F's
tutor answers, so no regeneration.

In [ ]:
!pip install -q -U sentence-transformers sentencepiece

import json, glob, re, os, time, collections
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# ---- config -------------------------------------------------------------
NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
EMBEDDER  = "sentence-transformers/LaBSE"
TOP_K     = 8        # facts tested per claim after retrieval
ENTAIL_TH = 0.50     # provisional; every score is saved so this can be swept
CONTRA_TH = 0.50
BATCH     = 64
# -------------------------------------------------------------------------

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("eval")
WORK.mkdir(parents=True, exist_ok=True)


def find(name, *fallbacks):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for f in fallbacks:
        hits += glob.glob(f)
    if not hits:
        raise SystemExit(f"{name} not found — attach the dataset")
    return hits[0]


T = pd.read_csv(find("biology_all_triples.csv", "kg/triples/biology_all_triples.csv",
                     "../kg/triples/biology_all_triples.csv"))
T["subject"] = T.subject.astype(str).str.strip()
T["object"] = T.object.astype(str).str.strip()

# Run F's claims, minus its verdicts — this notebook re-decides them.
C = pd.read_csv(find("claim_verdicts.csv", "eval/run_F/claim_verdicts.csv",
                     "../eval/run_F/claim_verdicts.csv"))
C = C[["model", "qid", "claim_no", "chapter_no", "question", "claim"]].drop_duplicates()
C = C.reset_index(drop=True)
print(f"{len(T)} facts | {len(C)} claims from {C.model.nunique()} models")

## 1 — Verbalise triples

One template per relation. An NLI model reads sentences, and the relation's meaning has to
be carried by Bangla grammar rather than by a bare label sitting between two strings.

In [ ]:
TEMPLATES = {
    "সংজ্ঞা":   "{s} হলো {o}।",
    "অংশ":      "{s} এর অংশ হলো {o}।",
    "প্রকার":    "{o} হলো {s} এর একটি প্রকার।",
    "কাজ":      "{s} এর কাজ হলো {o}।",
    "অবস্থান":   "{s} {o} এ অবস্থিত।",
    "কারণ":     "{s} এর কারণ হলো {o}।",
    "লক্ষণ":     "{s} এর একটি লক্ষণ হলো {o}।",
    "প্রতিরোধ":  "{s} প্রতিরোধে {o} প্রয়োজন।",
    "পরিমাণ":   "{s} এর পরিমাণ {o}।",
}


def verbalise(s, rel, o):
    return TEMPLATES.get(rel, "{s} এর {r} হলো {o}।".replace("{r}", rel)).format(s=s, o=o)


T["sentence"] = [verbalise(r.subject, r.relation, r.object) for r in T.itertuples()]
print("examples:")
for r in T.drop_duplicates("relation").head(9).itertuples():
    print(f"  [{r.relation}] {r.sentence[:95]}")

## 2 — Retrieve candidate facts

Entity-linked facts first, topped up by LaBSE nearest neighbours. Testing all 2,248 facts
against all claims would be 3.1M entailment calls; this keeps it near 11k.

In [ ]:
from sentence_transformers import SentenceTransformer

WORD = re.compile(r"[ঀ-৿]+|[A-Za-z]+|[0-9০-৯]+")
SUFFIXES = ["গুলোর", "গুলোকে", "গুলো", "টিকে", "গুলি", "দের", "টির", "টি",
            "য়ের", "এর", "কে", "ের", "রা", "র"]
GENERIC = {"মাধ্যম", "ধরন", "জিনিস", "সময়", "স্থান", "গুরুত্ব", "নাম",
           "ব্যাপার", "কারণ", "ফলে", "দিক", "অবস্থা", "অংশ"}


def lemma(w):
    for s in SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 3:
            return w[: -len(s)]
    return w


def toks(s):
    return [lemma(w) for w in WORD.findall(str(s))]


by_len = collections.defaultdict(dict)
for e in T.subject.unique():
    tk = tuple(toks(e))
    if tk and len("".join(tk)) >= 4 and e not in GENERIC:
        by_len[len(tk)][tk] = e
MAXN = max(by_len)
FACT_IDX = collections.defaultdict(list)
for i, r in enumerate(T.itertuples()):
    FACT_IDX[r.subject].append(i)


def link(text):
    tk, found, covered = toks(text), [], set()
    for n in range(MAXN, 0, -1):
        tab = by_len.get(n)
        if not tab:
            continue
        for i in range(len(tk) - n + 1):
            if any(j in covered for j in range(i, i + n)):
                continue
            hit = tab.get(tuple(tk[i:i + n]))
            if hit:
                found.append(hit)
                covered.update(range(i, i + n))
    return found


dev = "cuda" if torch.cuda.is_available() else "cpu"
emb = SentenceTransformer(EMBEDDER, device=dev)
F_emb = emb.encode(T.sentence.tolist(), batch_size=128, convert_to_numpy=True,
                   normalize_embeddings=True, show_progress_bar=False)
C_emb = emb.encode(C.claim.astype(str).tolist(), batch_size=128, convert_to_numpy=True,
                   normalize_embeddings=True, show_progress_bar=False)

sims = C_emb @ F_emb.T
pairs = []
for ci, r in enumerate(C.itertuples()):
    linked = {i for e in link(r.claim) for i in FACT_IDX.get(e, [])}
    nn = np.argsort(-sims[ci])[:TOP_K]
    cand = list(linked)[:TOP_K] + [int(j) for j in nn if j not in linked]
    for fi in cand[:TOP_K]:
        pairs.append((ci, fi, float(sims[ci, fi])))

P = pd.DataFrame(pairs, columns=["ci", "fi", "sim"])
print(f"{len(P)} (claim, fact) pairs to test — {len(P)/len(C):.1f} per claim")
del F_emb, C_emb, sims, emb
torch.cuda.empty_cache()

## 3 — Entailment

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(dev).eval()
lab = {i: v.lower() for i, v in nli.config.id2label.items()}
ENT = next(i for i, v in lab.items() if "entail" in v)
CON = next(i for i, v in lab.items() if "contra" in v)
print(f"NLI labels: {lab}")

prem = T.sentence.values
hyp = C.claim.astype(str).values
ent_p, con_p = np.zeros(len(P)), np.zeros(len(P))

t0 = time.time()
with torch.inference_mode():
    for s in range(0, len(P), BATCH):
        blk = P.iloc[s:s + BATCH]
        enc = tok([prem[i] for i in blk.fi], [hyp[i] for i in blk.ci],
                  return_tensors="pt", padding=True, truncation=True, max_length=256).to(dev)
        pr = nli(**enc).logits.softmax(-1).cpu().numpy()
        ent_p[s:s + len(blk)] = pr[:, ENT]
        con_p[s:s + len(blk)] = pr[:, CON]
        if s % (BATCH * 20) == 0:
            print(f"  {s}/{len(P)}  {time.time()-t0:.0f}s", end="\r")

P["entail"] = ent_p
P["contradict"] = con_p
print(f"\n{len(P)} pairs scored in {time.time()-t0:.0f}s")

## 4 — Aggregate to a verdict

Entailment takes precedence: a claim the curriculum supports is supported even if some
other fact about the same entity conflicts. Contradiction is checked only in its absence.

In [ ]:
g = P.groupby("ci")
best_e = g.entail.idxmax()
best_c = g.contradict.idxmax()

res = pd.DataFrame({
    "ci": best_e.index,
    "max_entail": P.loc[best_e, "entail"].values,
    "entail_fi": P.loc[best_e, "fi"].values,
    "max_contradict": P.loc[best_c, "contradict"].values,
    "contradict_fi": P.loc[best_c, "fi"].values,
}).set_index("ci")

C = C.join(res)
C["verdict"] = np.where(C.max_entail >= ENTAIL_TH, "supported",
                np.where(C.max_contradict >= CONTRA_TH, "contradicted", "not_in_curriculum"))
C["evidence"] = [T.sentence.iloc[int(i)] if pd.notna(i) else ""
                 for i in np.where(C.verdict == "contradicted", C.contradict_fi, C.entail_fi)]
C["system"] = "G_nli"
C.to_csv(WORK / "nli_claim_verdicts.csv", index=False)

print(C.verdict.value_counts().to_string())
print("\nby model (%):")
print(pd.crosstab(C.model, C.verdict, normalize="index").mul(100).round(1).to_string())

print("\nthreshold sweep:")
print(f"  {'thresh':>7}  {'supported':>10}  {'contradicted':>13}  {'neither':>9}")
for th in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    sup = (C.max_entail >= th).mean() * 100
    con = ((C.max_entail < th) & (C.max_contradict >= th)).mean() * 100
    print(f"  {th:>7.2f}  {sup:>9.1f}%  {con:>12.1f}%  {100-sup-con:>8.1f}%")

## 5 — Against the earlier systems

In [ ]:
old = pd.read_csv(find("all_system_verdicts.csv", "eval/run_F/all_system_verdicts.csv",
                       "../eval/run_F/all_system_verdicts.csv"))
cols = ["model", "qid", "claim_no", "claim", "verdict", "system"]
ALL = pd.concat([old[cols], C[cols]], ignore_index=True)
ALL.to_csv(WORK / "all_system_verdicts_with_nli.csv", index=False)

print("verdict distribution by system (%):")
print(pd.crosstab(ALL.system, ALL.verdict, normalize="index").mul(100).round(1).to_string())

wide = ALL.pivot_table(index=["model", "qid", "claim_no"], columns="system",
                       values="verdict", aggfunc="first").dropna()
print(f"\nclaims judged by every system: {len(wide)}")
for a in ["D_graph", "E1_closed_book", "E2_retrieval"]:
    if a in wide.columns and "G_nli" in wide.columns:
        print(f"  G_nli vs {a}: agree {(wide['G_nli'] == wide[a]).mean()*100:.1f}%")

In [ ]:
for v in ["supported", "contradicted"]:
    sub = C[C.verdict == v]
    print(f"\n{'='*80}\n{v}  ({len(sub)} claims)\n{'='*80}")
    for r in sub.head(4).itertuples():
        sc = r.max_entail if v == "supported" else r.max_contradict
        print(f"\n  claim   : {str(r.claim)[:100]}")
        print(f"  evidence: {str(r.evidence)[:100]}")
        print(f"  score   : {sc:.3f}")

# Labelling sample, weighted to where NLI and the baselines disagree.
dis = wide[wide.G_nli != wide.E1_closed_book].reset_index()
agr = wide[wide.G_nli == wide.E1_closed_book].reset_index()
S = pd.concat([dis.sample(min(150, len(dis)), random_state=17),
               agr.sample(min(50, len(agr)), random_state=17)])
S = S.merge(C[["model", "qid", "claim_no", "claim", "question", "evidence",
               "max_entail", "max_contradict"]],
            on=["model", "qid", "claim_no"], how="left").reset_index(drop=True)
S.insert(0, "triple_id", [f"N{i:04d}" for i in range(1, len(S) + 1)])
S["subject"] = S.claim
S["relation"] = S.G_nli
S["object"] = S.evidence.fillna("")
S["source_text"] = ("প্রশ্ন: " + S.question.fillna("").astype(str)
                    + "   |   entail " + S.max_entail.round(2).astype(str)
                    + " / contradict " + S.max_contradict.round(2).astype(str))
S[["triple_id", "model", "subject", "relation", "object", "source_text"]].to_csv(
    WORK / "nli_verdict_review_sample.csv", index=False)
print(f"\n\n{len(S)} verdicts to label -> nli_verdict_review_sample.csv")
print("1 = verdict right, 3 = wrong, 2 = right call but wrong evidence.")